# USDA FoodData Central — Nutritional Data Engineering & Preprocessing Pipeline
**Author:** Kristian Hudák  
**Competition:** National Olympiad in Artificial Intelligence (OAI) — Home Qualifier Round

This notebook processes the raw USDA Foundation Food database into clean, structured feature tables for downstream machine learning tasks:
1. Ingestion and unpacking of official USDA Foundation Food CSV tables (`food.csv`, `nutrient.csv`, `food_nutrient.csv`)
2. Filtering and joining key macronutrient IDs (energy, protein, carbohydrate, total lipid)
3. Computing derived nutritional ratios ($C/P$, $C/Carb$, $C/Fat$)
4. Merging food category codes and standardized ingredient descriptions
5. Exporting pivoted tabular datasets (`pivoted_categorized_food_data.xlsx` and `df_fdcid_name_calories.xlsx`)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
# Select computation device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)


cpu


In [ ]:
# Ingest and extract USDA Foundation Food zip archive
!wget /content/FoodData_Central_foundation_food_csv_2025-04-24.zip
!unzip -o FoodData_Central_foundation_food_csv_2025-04-24.zip


/content/FoodData_Central_foundation_food_csv_2025-04-24.zip: Scheme missing.
Archive:  FoodData_Central_foundation_food_csv_2025-04-24.zip
  inflating: FoodData_Central_foundation_food_csv_2025-04-24/food_update_log_entry.csv  
  inflating: FoodData_Central_foundation_food_csv_2025-04-24/agricultural_samples.csv  
  inflating: FoodData_Central_foundation_food_csv_2025-04-24/Download API Field Descriptions.xlsx  
  inflating: FoodData_Central_foundation_food_csv_2025-04-24/input_food.csv  
  inflating: FoodData_Central_foundation_food_csv_2025-04-24/food_attribute_type.csv  
  inflating: FoodData_Central_foundation_food_csv_2025-04-24/lab_method_code.csv  
  inflating: FoodData_Central_foundation_food_csv_2025-04-24/measure_unit.csv  
  inflating: FoodData_Central_foundation_food_csv_2025-04-24/food_nutrient_conversion_factor.csv  
  inflating: FoodData_Central_foundation_food_csv_2025-04-24/nutrient.csv  
  inflating: FoodData_Central_foundation_food_csv_2025-04-24/sub_sample_food.csv

### Exploratory Data Analysis & Table Schema Verification


In [ ]:
data1=pd.read_csv('/content/FoodData_Central_foundation_food_csv_2025-04-24/food.csv')
data1.head(100000)


,fdc_id,data_type,description,food_category_id,publication_date
0,319874,sample_food,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
1,319875,market_acquisition,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
2,319876,market_acquisition,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
3,319877,sub_sample_food,Hummus,16.0,2019-04-01
4,319878,sub_sample_food,Hummus,16.0,2019-04-01
...,...,...,...,...,...
74170,2731069,sub_sample_food,"Squash, yellow, raw",NaN,2024-01-18
74171,2731070,sub_sample_food,"Squash, yellow, raw",NaN,2024-01-18
74172,2731071,sub_sample_food,"Squash, yellow, raw",NaN,2024-01-18
74173,2731072,sub_sample_food,"Squash, yellow, raw",NaN,2024-01-18


In [ ]:
data1[data1['fdc_id'] == 321358]


,fdc_id,data_type,description,food_category_id,publication_date
651,321358,foundation_food,"Hummus, commercial",16.0,2019-04-01


In [ ]:
data1['description'].value_counts()


,count
description,
"SOY MILK, UNSWEETENED, PLAIN, SHELF STABLE",2272
"ALMOND MILK, UNSWEETENED, PLAIN, SHELF STABLE",1858
"OIL, SUNFLOWER",628
"OIL, SAFFLOWER",628
"OIL, OLIVE, EXTRA LIGHT",626
...,...
"Romaine Lettuce, Region 2, n/a, Yes - Minerals - NFY0101R6",1
"Romaine Lettuce, Region 2, n/a, Yes - Sugars - NFY0101R7",1
"Romaine Lettuce, Region 2, n/a, Yes - Starch - NFY0101R8",1


In [ ]:
len(data1)


74175

In [ ]:
data2=pd.read_csv('/content/FoodData_Central_foundation_food_csv_2025-04-24/nutrient.csv')
data2.head(4000)


,id,name,unit_name,nutrient_nbr,rank
0,2047,Energy (Atwater General Factors),KCAL,957.0,280.0
1,2048,Energy (Atwater Specific Factors),KCAL,958.0,290.0
2,1001,Solids,G,201.0,200.0
3,1002,Nitrogen,G,202.0,500.0
4,1003,Protein,G,203.0,600.0
...,...,...,...,...,...
472,2064,Oligosaccharides,MG,NaN,2250.0
473,2065,Low Molecular Weight Dietary Fiber (LMWDF),G,293.4,1306.0
474,2068,Vitamin E,MG,959.0,7810.0
475,2067,Vitamin A,UG,960.0,7430.0


In [ ]:
data3=pd.read_csv('/content/FoodData_Central_foundation_food_csv_2025-04-24/food_nutrient.csv')
data3.head(4000)


/tmp/ipython-input-473613737.py:1: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  data3=pd.read_csv('/content/FoodData_Central_foundation_food_csv_2025-04-24/food_nutrient.csv')


,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired
0,2201847,319877,1051,56.300,1.0,1.0,NaN,NaN,NaN,NaN,NaN
1,2201845,319877,1002,1.280,1.0,1.0,NaN,NaN,NaN,NaN,NaN
2,2201846,319877,1004,19.000,1.0,1.0,NaN,NaN,NaN,NaN,NaN
3,2201844,319877,1007,1.980,1.0,1.0,NaN,NaN,NaN,NaN,NaN
4,2201852,319878,1091,188.000,1.0,1.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
3995,2221370,321900,1085,0.070,NaN,4.0,NaN,NaN,NaN,NaN,NaN
3996,2221288,321900,1216,0.028,4.0,1.0,0.025,0.03,0.028,NaN,2001.0
3997,2221336,321900,1211,0.088,4.0,1.0,0.072,0.11,0.086,NaN,2001.0
3998,2221277,321900,1009,0.000,1.0,1.0,NaN,NaN,0.000,NaN,2001.0


In [ ]:
len(data3)


155243

In [ ]:
# Drop redundant metadata columns to optimize memory
data_short=data3.drop(columns=['id','data_points','derivation_id','min','max','median','footnote','min_year_acquired'])
data_short.head(10)


,fdc_id,nutrient_id,amount
0,319877,1051,56.30
1,319877,1002,1.28
2,319877,1004,19.00
3,319877,1007,1.98
4,319878,1091,188.00
5,319878,1101,1.21
6,319878,1092,326.00
7,319878,1087,46.00
8,319878,1093,446.00
9,319878,1090,75.90


In [ ]:
# Filter food items containing complete macronutrient profiles (calories, carbs, fat, protein)sanie do listu

target1 = {2047, 1003, 1005,1004}

filtered = data_short[data_short["nutrient_id"].isin(target1)]

valid_fdc_ids = (
    filtered.groupby("fdc_id")["nutrient_id"]
    .apply(set)
    .reset_index()
)

valid_fdc_ids = valid_fdc_ids[valid_fdc_ids["nutrient_id"].apply(lambda x: target1.issubset(x))]

print(valid_fdc_ids["fdc_id"].tolist())
len(valid_fdc_ids["fdc_id"].tolist())


[321358, 321360, 321611, 323121, 323294, 323505, 323604, 323697, 323793, 324157, 324317, 324653, 324860, 325036, 325198, 325287, 325430, 325524, 325871, 326196, 326698, 327046, 327357, 328637, 328841, 329370, 329490, 329596, 329716, 330137, 330415, 330458, 331897, 331960, 332282, 332397, 332791, 333008, 333281, 333374, 333476, 334194, 334536, 334628, 334720, 335240, 746758, 746759, 746760, 746761, 746762, 746763, 746764, 746765, 746766, 746767, 746768, 746769, 746770, 746771, 746772, 746773, 746774, 746776, 746777, 746778, 746779, 746780, 746782, 746783, 746784, 746785, 746952, 747429, 747447, 747448, 747693, 747997, 748236, 748967, 749420, 789890, 789951, 790018, 790085, 790146, 790214, 790276, 790577, 790646, 1104647, 1104705, 1104766, 1104812, 1104867, 1104913, 1104962, 1105073, 1105314, 1105430, 1105547, 1105664, 1105781, 1105897, 1750337, 1750338, 1750339, 1750340, 1750341, 1750342, 1750343, 1750344, 1750345, 1750346, 1750347, 1750352, 1750353, 1750354, 1999626, 1999627, 1999628, 

321

In [ ]:
# Keep only rows with all 4 target nutrient identifiers
target2 = [2047, 1003, 1005,1004]

filtered_data = data_short[data_short["fdc_id"].isin(valid_fdc_ids["fdc_id"])]
#zmena tabulky tak aby bola kazda nutrient v vlastnej column
pivoted = filtered_data.pivot(
    index="fdc_id",
    columns="nutrient_id",
    values="amount"
).reset_index()
pivoted.columns.name = None

pivoted = pivoted[["fdc_id"] + target2]   # nechá len tieto 5 stĺpce

pivoted.columns = ["fdc_id", "calories", "protein", "carbs",'fat'] #premenovanie columns

#zmenenie negativych hodnot ktore su chyby v datasete na nulove hodnoty ktore davaju zmysel
cols = ["protein", "fat", "carbs","calories"]
pivoted[cols] = pivoted[cols].clip(lower=0)
pivoted


,fdc_id,calories,protein,carbs,fat
0,321358,243.0000,7.35000,14.90000,17.100
1,321360,31.0000,0.83000,5.51000,0.630
2,321611,24.0000,1.04000,4.11000,0.390
3,323121,310.0000,11.70000,2.89000,28.000
4,323294,667.0000,20.40000,16.20000,57.800
...,...,...,...,...,...
316,2727572,190.0448,21.32500,0.22120,11.540
317,2727573,143.2000,21.09375,0.18025,6.456
318,2727574,140.1440,21.97500,0.21800,5.708
319,2727575,138.2750,22.81250,0.00000,5.475


In [ ]:
# Feature engineering: calculate calorie-to-macronutrient density ratios
pivoted['C/P'] = pivoted.apply(
    lambda row: row['calories']/row['protein'] if row['protein'] != 0 else row['calories'],
    axis=1
)
pivoted['C/Cb'] = pivoted.apply(
    lambda row: row['calories']/row['carbs'] if row['carbs'] != 0 else row['calories'],
    axis=1
)
pivoted['C/F'] = pivoted.apply(
    lambda row: row['calories']/row['fat'] if row['fat'] != 0 else row['calories'],
    axis=1
)
pivoted


,fdc_id,calories,protein,carbs,fat,C/P,C/Cb,C/F
0,321358,243.0000,7.35000,14.90000,17.100,33.061224,16.308725,14.210526
1,321360,31.0000,0.83000,5.51000,0.630,37.349398,5.626134,49.206349
2,321611,24.0000,1.04000,4.11000,0.390,23.076923,5.839416,61.538462
3,323121,310.0000,11.70000,2.89000,28.000,26.495726,107.266436,11.071429
4,323294,667.0000,20.40000,16.20000,57.800,32.696078,41.172840,11.539792
...,...,...,...,...,...,...,...,...
316,2727572,190.0448,21.32500,0.22120,11.540,8.911831,859.153707,16.468354
317,2727573,143.2000,21.09375,0.18025,6.456,6.788741,794.452150,22.180917
318,2727574,140.1440,21.97500,0.21800,5.708,6.377429,642.862385,24.552207
319,2727575,138.2750,22.81250,0.00000,5.475,6.061370,138.275000,25.255708


In [ ]:
pivoted_categorized = pivoted.merge(
    data1[['fdc_id', 'food_category_id']],
    on='fdc_id',
    how='left'
)
pivoted_categorized


,fdc_id,calories,protein,carbs,fat,C/P,C/Cb,C/F,food_category_id
0,321358,243.0000,7.35000,14.90000,17.100,33.061224,16.308725,14.210526,16.0
1,321360,31.0000,0.83000,5.51000,0.630,37.349398,5.626134,49.206349,11.0
2,321611,24.0000,1.04000,4.11000,0.390,23.076923,5.839416,61.538462,11.0
3,323121,310.0000,11.70000,2.89000,28.000,26.495726,107.266436,11.071429,7.0
4,323294,667.0000,20.40000,16.20000,57.800,32.696078,41.172840,11.539792,12.0
...,...,...,...,...,...,...,...,...,...
316,2727572,190.0448,21.32500,0.22120,11.540,8.911831,859.153707,16.468354,13.0
317,2727573,143.2000,21.09375,0.18025,6.456,6.788741,794.452150,22.180917,13.0
318,2727574,140.1440,21.97500,0.21800,5.708,6.377429,642.862385,24.552207,13.0
319,2727575,138.2750,22.81250,0.00000,5.475,6.061370,138.275000,25.255708,10.0


In [ ]:
# Join food category IDs and natural language descriptions for downstream optimization
pivoted_categorized = pivoted_categorized.merge(
    data1[['fdc_id', 'description']],
    on='fdc_id',
    how='left'
)
pivoted_categorized


,fdc_id,calories,protein,carbs,fat,C/P,C/Cb,C/F,food_category_id,description
0,321358,243.0000,7.35000,14.90000,17.100,33.061224,16.308725,14.210526,16.0,"Hummus, commercial"
1,321360,31.0000,0.83000,5.51000,0.630,37.349398,5.626134,49.206349,11.0,"Tomatoes, grape, raw"
2,321611,24.0000,1.04000,4.11000,0.390,23.076923,5.839416,61.538462,11.0,"Beans, snap, green, canned, regular pack, drai..."
3,323121,310.0000,11.70000,2.89000,28.000,26.495726,107.266436,11.071429,7.0,"Frankfurter, beef, unheated"
4,323294,667.0000,20.40000,16.20000,57.800,32.696078,41.172840,11.539792,12.0,"Nuts, almonds, dry roasted, with salt added"
...,...,...,...,...,...,...,...,...,...,...
316,2727572,190.0448,21.32500,0.22120,11.540,8.911831,859.153707,16.468354,13.0,"Beef, short loin (NY strip steak), raw"
317,2727573,143.2000,21.09375,0.18025,6.456,6.788741,794.452150,22.180917,13.0,"Beef, tenderloin steak, raw"
318,2727574,140.1440,21.97500,0.21800,5.708,6.377429,642.862385,24.552207,13.0,"Beef, top sirloin steak, raw"
319,2727575,138.2750,22.81250,0.00000,5.475,6.061370,138.275000,25.255708,10.0,"Pork, chop, center cut, raw"


In [ ]:
# Export processed dataset for downstream modeling


In [ ]:
pivoted_categorized.to_excel("pivoted_categorized_food_data.xlsx", index=False)


In [ ]:
from google.colab import files
files.download("pivoted_categorized_food_data.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
pivoted_categorized.describe()


,fdc_id,calories,protein,carbs,fat,C/P,C/Cb,C/F,food_category_id
count,3.210000e+02,321.000000,321.000000,321.000000,321.000000,321.000000,321.000000,321.000000,321.000000
mean,1.831626e+06,194.281280,9.500428,19.501433,8.709333,54.805818,56.675759,129.479275,11.137072
std,8.926429e+05,184.265232,10.157294,25.203117,15.626189,105.006122,132.233539,148.071684,5.619711
min,3.213580e+05,14.000000,0.000000,0.000000,0.000000,4.106449,4.026104,9.031282,1.000000
25%,7.494200e+05,46.395100,1.039375,3.270000,0.287500,14.160414,4.771756,18.362069,9.000000
50%,2.261420e+06,117.000000,6.693750,8.160000,1.490000,25.881980,5.913560,89.527469,11.000000
75%,2.644291e+06,357.064800,16.468750,20.196825,8.880000,46.808511,41.172840,178.947368,15.000000
max,2.727576e+06,895.000000,79.900000,99.600000,99.100000,895.000000,1065.476190,1253.125000,25.000000


### Formatting Dataset for Task 1 (Caloric Prediction)


In [ ]:
# Extract simplified feature subsets for calorie regression
df_fdcid_calories=pivoted.drop(columns=['protein','carbs','fat','C/P','C/Cb','C/F'])
df_fdcid_calories


,fdc_id,calories
0,321358,243.0000
1,321360,31.0000
2,321611,24.0000
3,323121,310.0000
4,323294,667.0000
...,...,...
316,2727572,190.0448
317,2727573,143.2000
318,2727574,140.1440
319,2727575,138.2750


In [ ]:
# Join descriptions for ingredient fuzzy lookup
df_fdcid_name_calories = df_fdcid_calories.merge(
    data1[['fdc_id', 'description']],
    on='fdc_id',
    how='left'
)
df_fdcid_name_calories


,fdc_id,calories,description
0,321358,243.0000,"Hummus, commercial"
1,321360,31.0000,"Tomatoes, grape, raw"
2,321611,24.0000,"Beans, snap, green, canned, regular pack, drai..."
3,323121,310.0000,"Frankfurter, beef, unheated"
4,323294,667.0000,"Nuts, almonds, dry roasted, with salt added"
...,...,...,...
316,2727572,190.0448,"Beef, short loin (NY strip steak), raw"
317,2727573,143.2000,"Beef, tenderloin steak, raw"
318,2727574,140.1440,"Beef, top sirloin steak, raw"
319,2727575,138.2750,"Pork, chop, center cut, raw"


In [ ]:
# Save final processed dataset to Excel
df_fdcid_name_calories.to_excel("df_fdcid_name_calories.xlsx", index=False)

print("Excel súbor uložený:","df_fdcid_name_calories.xlsx")


Excel súbor uložený: df_fdcid_name_calories.xlsx


In [ ]:
from google.colab import files
files.download("df_fdcid_name_calories.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>